In [1]:
# Numpy
import numpy as np
# Plotting stuff and suppressing warnings (dividing by zero)
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')
import sys

# Import Astropy libraries - useful for many astronomy related function
from astropy.table import Table
from astropy.io import fits
#import fitsio

# Settings for matplotlib plots
settings = {
    'font.size':12,
    'axes.linewidth':2.0,
    'xtick.major.size':6.0,
    'xtick.minor.size':4.0,
    'xtick.major.width':2.0,
    'xtick.minor.width':1.,
    'xtick.direction':'in', 
    'xtick.minor.visible':True,
    'xtick.top':True,
    'ytick.major.size':6.0,
    'ytick.minor.size':4.0,
    'ytick.major.width':2.0,
    'ytick.minor.width':1.,
    'ytick.direction':'in', 
    'ytick.minor.visible':True,
    'ytick.right':True,
    'axes.labelsize':14
}

plt.rcParams.update(**settings)

In [2]:
def get_agn_maskbits(file):
    import yaml
    from desiutil_bitmask import BitMask
    file_yaml = open(file, 'r')
    yaml_defs = yaml.safe_load(file_yaml)
    
    AGN_MASKBITS = BitMask('AGN_MASKBITS', yaml_defs)
    OPT_UV_TYPE = BitMask('OPT_UV_TYPE', yaml_defs)
    IR_TYPE = BitMask('IR_TYPE', yaml_defs)
    
    return AGN_MASKBITS, OPT_UV_TYPE, IR_TYPE

In [3]:
# Set the directory of the catalog file here:
#catdir = '/global/cfs/cdirs/desi/public/dr1/vac/dr1/agnqso/v1.0/'  # NERSC
catdir = './' # Local copy of VAC file (**NOTE**: Edit as needed if you move your files around)

In [4]:
# Open the catalog
agn_hdul = fits.open(f'{catdir}agnqso_desi.fits', format='fits')
agn_hdul.info()

Filename: ./agnqso_desi.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  AGNCAT        1 BinTableHDU     89   17995599R x 36C   [K, 7A, 6A, J, D, D, K, 6A, K, K, K, J, D, D, K, D, D, D, I, E, L, L, L, K, K, K, K, K, K, K, K, K, K, K, K, K]   
  2  AUXDATA       1 BinTableHDU    182   17995599R x 58C   [K, 7A, 6A, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E]   


In [ ]:
#%%time
from astropy.io import fits
from astropy.table import Table


with fits.open('./agnqso_desi.fits', memmap=True) as hdul:
    data1 = hdul[1].data
    
    # Defining only the columns needed
    cols_to_keep = ['TARGETID', 'Z', 'OPT_UV_TYPE', 'TARGET_RA', 'TARGET_DEC']
    
    
    T = Table({col: data1[col] for col in cols_to_keep})

print(f"Success! Loaded {len(T)} objects into T safely.")

Success! Loaded 17995599 objects into T safely.


In [ ]:
from astropy.table import Table, join 

sl_vac = Table.read('./desi-sl-vac-v1.fits')

# Join data from both catalogs
lensed_candidates = join(T, sl_vac, keys='TARGETID')

print(f"Success! Found {len(lensed_candidates)} matches.")



lensed_candidates.show_in_notebook(display_adapter='html')



Success! Found 2277 matches.


DataGrid(auto_fit_params={'area': 'all', 'padding': 30, 'numCols': None}, corner_renderer=None, default_render…

In [15]:
lensed_candidates.write('DESI_SL_Matches.csv', format='ascii.csv', overwrite=True)

In [16]:
# 1. Select only the specific columns you need
final_table = lensed_candidates['TARGETID', 'Z_1', 'TARGET_RA_1', 'TARGET_DEC_1', 'SURVEY']

# 2. Export to a CSV file on your D: drive
# Using 'ascii.csv' makes it instantly readable by Excel
final_table.write('DESI_Lensing_Candidates.csv', format='ascii.csv', overwrite=True)

print(f"Successfully exported {len(final_table)} candidates to Excel-compatible CSV.")

Successfully exported 2277 candidates to Excel-compatible CSV.
